In [1]:
from pathlib import Path

import numpy as np
import pyvista as pv

from ulac import mapper
from ulac.common import serialization, visualization

In [ ]:
target_mesh_dir = Path("../data/patient_01")
source_patient_ids = (
    "patient_02",
    "patient_03",
    "patient_04",
    "patient_05",
    "patient_06",
    "patient_07",
)
original_mesh_file_name = "mesh_with_fibers_tags.vtu"
uac_mesh_file_name = "uac_submesh_data.msgpack"

In [3]:
target_mesh_original = pv.read(target_mesh_dir / original_mesh_file_name)
target_mesh_uac = serialization.load_uac_submeshdata(target_mesh_dir / uac_mesh_file_name)
target_mesh_fibers = np.array(target_mesh_original.cell_data["fibers"])
target_uac_mapper = mapper.UACMapper(
    target_mesh_original.points, target_mesh_original.n_cells, target_mesh_uac
)

target_mesh_copy = target_mesh_original.copy()
target_mesh_copy.clear_cell_data()
target_mesh_copy.clear_point_data()
np.save(target_mesh_dir / "fiber_field.npy", target_mesh_fibers)
pv.save_meshio(target_mesh_dir / "mesh.vtu", target_mesh_copy)

In [4]:
fiber_fields_mapped_to_target = dict.fromkeys(source_patient_ids)
for patient_id in source_patient_ids:
    source_mesh_dir = Path(f"../data/{patient_id}")
    source_mesh_original = pv.read(source_mesh_dir / original_mesh_file_name)
    source_mesh_uac = serialization.load_uac_submeshdata(source_mesh_dir / uac_mesh_file_name)
    source_mesh_fibers = np.array(source_mesh_original.cell_data["fibers"])
    source_uac_mapper = mapper.UACMapper(
        source_mesh_original.points, source_mesh_original.n_cells, source_mesh_uac
    )
    fibers_source_uac_mesh = source_uac_mapper.map_vector_from_original_to_uac_mesh(
        source_mesh_fibers
    )
    fibers_target_uac_mesh = source_uac_mapper.interpolate_to_other_uac_mesh(
        fibers_source_uac_mesh, target_uac_mapper
    )
    target_mesh_fibers = target_uac_mapper.map_vector_from_uac_to_original_mesh(
        fibers_target_uac_mesh
    )
    fiber_fields_mapped_to_target[patient_id] = target_mesh_fibers

np.savez(target_mesh_dir / "fiber_fields_mapped_to_target.npz", **fiber_fields_mapped_to_target)

FileNotFoundError: File (/home/max/4-Code/ulac/data/partient_02/mesh_with_fibers_tags.vtu) not found

In [ ]:
visualization.plot_mesh_with_vector_field(
    source_mesh_original,
    source_mesh_original.cell_data["fibers"],
    scaling_factor=500,
    vector_color="blue",
)

visualization.plot_uac_mesh_with_vector_field(
    source_mesh_uac,
    fibers_source_uac_mesh,
    scaling_factor=0.02,
    vector_color="blue",
    mesh_color="lightgray",
)

visualization.plot_uac_mesh_with_vector_field(
    target_mesh_uac,
    fibers_target_uac_mesh,
    scaling_factor=0.02,
    vector_color="green",
    mesh_color="lightgray",
)

visualization.plot_mesh_with_vector_field(
    target_mesh_original,
    target_mesh_fibers,
    scaling_factor=500,
    vector_color="green",
    mesh_color="lightgray",
)